# 16 — Arm seimbang dari generator tekstur terpilih vs real_only
Input: paket kode, dataset regrouped v4, assets, `review_p100.csv`, dan output kernel 13.
Menjalankan `select` (terkunci pada hash review), membangun SATU arm: 301 foto real + kandidat accepted pemenang berperingkat terbaik (jarak statistik tekstur ke unripe nyata) sampai defisit 133, lalu YOLOv8n 3000 update, eval tiap 250, seed 0–4 — identik dengan eksperimen balancing C sehingga real_only/oversample_r100/cyclegan_r100 C menjadi pembanding berpasangan. Hanya validation; test tidak dibaca. Permintaan pengguna 28 Sep 2026; menyimpang dari build-detectors pra-registrasi (hash order, k bersama G0, lima arm).


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
WORK = Path('/kaggle/working/texture_ranked_v1')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
# Kaggle unpacks an uploaded zip into plain files; accept either form, never both.
archives = list(INPUT.rglob('texture_study_code.zip'))
# A previous stage's kernel output also carries a code copy next to study/; only the code dataset counts.
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json')
            if (p.parent/'code/cyclegan/tools/texture_study.py').exists() and not (p.parent/'study').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)  # /kaggle/input is read-only
import hashlib
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
ENV = {**os.environ, 'PYTHONPATH':str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'PYTHONUNBUFFERED':'1'}
def unique(marker):
    hits=sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits)!=1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
ASSETS = unique('reviewed_masks.json')
ENV.update(MELON_DATASET=str(DATASET),MELON_ASSETS=str(ASSETS))
os.environ.update(ENV)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=ENV,check=True)
CLI = HERE/'tools/texture_study.py'
import torch
print(torch.__version__,torch.cuda.is_available())


In [ ]:
# ultralytics pinned to the version of the balancing experiment (C) so the trainer is identical.
import importlib.util, importlib.metadata as meta
if importlib.util.find_spec('ultralytics') is None or meta.version('ultralytics') != '8.4.163':
    subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.163'],check=True)
for package in ('torch','torchvision','ultralytics'): print(package, meta.version(package))
if torch.cuda.device_count() < 1: raise RuntimeError('GPU required')
print('GPUs', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


In [ ]:
# Study from notebook 13 output + filled review (small dataset), then the locked selection.
import csv, hashlib
source=unique('study.json')
STUDY=WORK/'study'
shutil.copytree(source,STUDY)
for p in [STUDY,*STUDY.rglob('*')]: p.chmod(p.stat().st_mode|0o200)
overrides=list(INPUT.rglob('review_p100.csv'))
if len(overrides)!=1: raise ValueError(f'Attach exactly one review_p100.csv; found {overrides}')
target=STUDY/'p100/review/review.csv'
ids=lambda path:[r['blind_id'] for r in csv.DictReader(path.open(newline=''))]
if ids(overrides[0])!=ids(target): raise ValueError('review_p100.csv blind IDs/order differ from the study')
shutil.copyfile(overrides[0],target)
print('review sha256', hashlib.sha256(target.read_bytes()).hexdigest())
run(CLI,'select','--study',STUDY)
print((STUDY/'selection.json').read_text())


In [ ]:
# One balanced arm: best-ranked accepted winner candidates up to the ripe-unripe deficit.
ARMS=WORK/'arms'
run(HERE/'tools/build_ranked_texture_arm.py','--study',STUDY,'--dataset',DATASET,'--out',ARMS)
ARM=next(p.name for p in ARMS.iterdir() if p.is_dir())
manifest=json.loads((ARMS/ARM/'arm_manifest.json').read_text())
print(ARM, manifest['counts'], 'accepted', manifest['accepted_winner'], 'added', manifest['added'], 'full_balance', manifest['full_balance'])


In [ ]:
# Same trainer, schedule and seeds as experiment C; seeds split over the two T4s.
import shlex, time
PLAN={0:[0,1,2],1:[3,4]}
n_gpu=torch.cuda.device_count()
def chain(gpu,seeds):
    out=WORK/f'group{gpu}'
    argv=[sys.executable,str(HERE/'tools/train_detector_arms.py'),'--arms-dir',str(ARMS),'--arms',ARM,
          '--seeds',*map(str,seeds),'--out',str(out),'--updates','3000','--eval-every','250','--export']
    env=f"CUDA_VISIBLE_DEVICES={gpu if n_gpu>=2 else 0} PYTHONPATH={shlex.quote(str(HERE))}"
    log=open(WORK/f'detector_gpu{gpu}.log','w')
    return subprocess.Popen(env+' '+shlex.join(argv),shell=True,cwd=str(HERE),stdout=log,stderr=subprocess.STDOUT)
started=time.time()
if n_gpu>=2: codes=[p.wait() for p in [chain(0,PLAN[0]),chain(1,PLAN[1])]]
else: codes=[chain(0,PLAN[0]+PLAN[1]).wait()]
print('return codes',codes,'wall minutes',round((time.time()-started)/60,1))
for g in PLAN:
    log=WORK/f'detector_gpu{g}.log'
    if log.exists(): print(log.read_text()[-1500:])
if any(codes): raise SystemExit('detector training failed')


In [ ]:
# Collect validation results; keep the output small (no study copy, no arm images).
import statistics
OUT=WORK/'result'; OUT.mkdir()
results,selected={},{}
for g in PLAN:
    part=json.loads((WORK/f'group{g}'/'results.json').read_text())
    results.update(part['results']); selected.update(part['selected'])
    shutil.copytree(WORK/f'group{g}',OUT/f'group{g}',ignore=shutil.ignore_patterns('weights.pt','last*.pt'))
seeds=[0,1,2,3,4]
missing=[s for s in seeds if f'{ARM}@{s}' not in selected]
if missing: raise SystemExit(f'missing seeds {missing}')
def summary(table):
    keys=[f'{ARM}@{s}' for s in seeds]
    return dict(map50_95=statistics.fmean(table[k]['map50_95'] for k in keys),
                sd=statistics.pstdev(table[k]['map50_95'] for k in keys),
                ap_unripe=statistics.fmean(table[k]['per_class']['unripe']['ap50_95'] for k in keys),
                recall_unripe=statistics.fmean(table[k]['recall'][1] if len(table[k]['recall'])>1 else float('nan') for k in keys))
for name in ('selection.json',): shutil.copy(STUDY/name,OUT/name)
shutil.copy(ARMS/ARM/'arm_manifest.json',OUT/'arm_manifest.json'); shutil.copy(ARMS/ARM/'ranking.csv',OUT/'ranking.csv')
report=dict(arm=ARM,seeds=seeds,updates=3000,eval_every=250,results_final=results,results_selected=selected,
            summary_final=summary(results),summary_selected=summary(selected),split='valid',test_used=False)
(OUT/'results.json').write_text(json.dumps(report,indent=2)+'\n')
print(json.dumps({k:report[k] for k in ('summary_selected','summary_final')},indent=1))
shutil.rmtree(STUDY,ignore_errors=True); shutil.rmtree(ARMS,ignore_errors=True); shutil.rmtree(WORK/'code',ignore_errors=True)
for g in PLAN: shutil.rmtree(WORK/f'group{g}',ignore_errors=True)
print(sorted(str(p.relative_to(WORK)) for p in WORK.rglob('*') if p.is_file())[:40])
